# 00 — Inspect UCF Sports Dataset
**Goal:** confirm the dataset path, discover videos, inspect folder labels, and save a manifest.  
Run this notebook first. It does not load ViTPose.

In [3]:

from pathlib import Path
import pandas as pd

# Set the actual project directory (parent of notebooks)
PROJECT_DIR = Path(r"C:\Users\admin\ViTPose_Action_Recognition")

# Dataset location
DATA_DIR = PROJECT_DIR / "data" / "raw" / "ucf_sports"

VIDEO_EXTS = {".avi", ".mp4", ".mpeg", ".mpg", ".mov"}

print("Project folder:", PROJECT_DIR.resolve())
print("Project folder exists:", PROJECT_DIR.exists())
print("Dataset folder:", DATA_DIR.resolve())
print("Dataset folder exists:", DATA_DIR.exists())

assert PROJECT_DIR.exists(), "Project folder not found. Check PROJECT_DIR."
assert DATA_DIR.exists(), "Dataset folder not found. Check DATA_DIR."

# Find videos recursively
videos = sorted(
    p for p in DATA_DIR.rglob("*")
    if p.is_file() and p.suffix.lower() in VIDEO_EXTS
)

print("Videos found:", len(videos))

for p in videos[:20]:
    print(p.relative_to(DATA_DIR))


Project folder: C:\Users\admin\ViTPose_Action_Recognition
Project folder exists: True
Dataset folder: C:\Users\admin\ViTPose_Action_Recognition\data\raw\ucf_sports
Dataset folder exists: True
Videos found: 13320
ApplyEyeMakeup\v_ApplyEyeMakeup_g01_c01.avi
ApplyEyeMakeup\v_ApplyEyeMakeup_g01_c02.avi
ApplyEyeMakeup\v_ApplyEyeMakeup_g01_c03.avi
ApplyEyeMakeup\v_ApplyEyeMakeup_g01_c04.avi
ApplyEyeMakeup\v_ApplyEyeMakeup_g01_c05.avi
ApplyEyeMakeup\v_ApplyEyeMakeup_g01_c06.avi
ApplyEyeMakeup\v_ApplyEyeMakeup_g02_c01.avi
ApplyEyeMakeup\v_ApplyEyeMakeup_g02_c02.avi
ApplyEyeMakeup\v_ApplyEyeMakeup_g02_c03.avi
ApplyEyeMakeup\v_ApplyEyeMakeup_g02_c04.avi
ApplyEyeMakeup\v_ApplyEyeMakeup_g03_c01.avi
ApplyEyeMakeup\v_ApplyEyeMakeup_g03_c02.avi
ApplyEyeMakeup\v_ApplyEyeMakeup_g03_c03.avi
ApplyEyeMakeup\v_ApplyEyeMakeup_g03_c04.avi
ApplyEyeMakeup\v_ApplyEyeMakeup_g03_c05.avi
ApplyEyeMakeup\v_ApplyEyeMakeup_g03_c06.avi
ApplyEyeMakeup\v_ApplyEyeMakeup_g04_c01.avi
ApplyEyeMakeup\v_ApplyEyeMakeup_g04_c02.

In [4]:
# This assumes the parent folder is the class name, e.g. Walking/video1.avi.
# If all videos are in one folder, labels will be the folder name and must be fixed manually.
rows = []
for p in videos:
    relative = p.relative_to(DATA_DIR)
    label = relative.parts[0] if len(relative.parts) > 1 else "UNKNOWN"
    rows.append({"video_path": str(p.resolve()), "relative_path": str(relative),
                 "label": label, "extension": p.suffix.lower()})

manifest = pd.DataFrame(rows)
display(manifest.head(20))
print("\nVideos per folder-derived label:")
display(manifest["label"].value_counts().rename_axis("label").reset_index(name="videos"))

,video_path,relative_path,label,extension
0,C:\Users\admin\ViTPose_Action_Recognition\data...,ApplyEyeMakeup\v_ApplyEyeMakeup_g01_c01.avi,ApplyEyeMakeup,.avi
1,C:\Users\admin\ViTPose_Action_Recognition\data...,ApplyEyeMakeup\v_ApplyEyeMakeup_g01_c02.avi,ApplyEyeMakeup,.avi
2,C:\Users\admin\ViTPose_Action_Recognition\data...,ApplyEyeMakeup\v_ApplyEyeMakeup_g01_c03.avi,ApplyEyeMakeup,.avi
3,C:\Users\admin\ViTPose_Action_Recognition\data...,ApplyEyeMakeup\v_ApplyEyeMakeup_g01_c04.avi,ApplyEyeMakeup,.avi
4,C:\Users\admin\ViTPose_Action_Recognition\data...,ApplyEyeMakeup\v_ApplyEyeMakeup_g01_c05.avi,ApplyEyeMakeup,.avi
5,C:\Users\admin\ViTPose_Action_Recognition\data...,ApplyEyeMakeup\v_ApplyEyeMakeup_g01_c06.avi,ApplyEyeMakeup,.avi
6,C:\Users\admin\ViTPose_Action_Recognition\data...,ApplyEyeMakeup\v_ApplyEyeMakeup_g02_c01.avi,ApplyEyeMakeup,.avi
7,C:\Users\admin\ViTPose_Action_Recognition\data...,ApplyEyeMakeup\v_ApplyEyeMakeup_g02_c02.avi,ApplyEyeMakeup,.avi
8,C:\Users\admin\ViTPose_Action_Recognition\data...,ApplyEyeMakeup\v_ApplyEyeMakeup_g02_c03.avi,ApplyEyeMakeup,.avi
9,C:\Users\admin\ViTPose_Action_Recognition\data...,ApplyEyeMakeup\v_ApplyEyeMakeup_g02_c04.avi,ApplyEyeMakeup,.avi



Videos per folder-derived label:


,label,videos
0,CricketShot,167
1,TennisSwing,166
2,PlayingDhol,164
3,HorseRiding,164
4,PlayingCello,164
...,...,...
96,BreastStroke,101
97,PlayingViolin,100
98,PullUps,100
99,Skijet,100


In [5]:
OUT_DIR = PROJECT_DIR / "data" / "processed"
OUT_DIR.mkdir(parents=True, exist_ok=True)
manifest_path = OUT_DIR / "manifest.csv"
manifest.to_csv(manifest_path, index=False)
print("Saved manifest:", manifest_path.resolve())
print("IMPORTANT: If any label is UNKNOWN or all labels are the same, correct the labels in manifest.csv before extraction.")

Saved manifest: C:\Users\admin\ViTPose_Action_Recognition\data\processed\manifest.csv
IMPORTANT: If any label is UNKNOWN or all labels are the same, correct the labels in manifest.csv before extraction.


In [6]:

import pandas as pd
from pathlib import Path

PROJECT_DIR = Path(r"C:\Users\admin\ViTPose_Action_Recognition")
manifest_path = PROJECT_DIR / "data" / "processed" / "manifest.csv"

df = pd.read_csv(manifest_path)

print("Total videos:", len(df))
print("Number of classes:", df["label"].nunique())
print("Missing labels:", df["label"].isna().sum())
print("UNKNOWN labels:", (df["label"].astype(str) == "UNKNOWN").sum())
print("\nFirst 15 classes:")
print(df["label"].value_counts().head(15))
print("\nFirst 5 rows:")
display(df.head())


Total videos: 13320
Number of classes: 101
Missing labels: 0
UNKNOWN labels: 0

First 15 classes:
label
CricketShot          167
TennisSwing          166
PlayingDhol          164
HorseRiding          164
PlayingCello         164
BoxingPunchingBag    163
Drumming             161
ShavingBeard         161
Punch                160
BenchPress           160
PlayingGuitar        160
IceDancing           158
PlayingSitar         157
BandMarching         155
Bowling              155
Name: count, dtype: int64

First 5 rows:


,video_path,relative_path,label,extension
0,C:\Users\admin\ViTPose_Action_Recognition\data...,ApplyEyeMakeup\v_ApplyEyeMakeup_g01_c01.avi,ApplyEyeMakeup,.avi
1,C:\Users\admin\ViTPose_Action_Recognition\data...,ApplyEyeMakeup\v_ApplyEyeMakeup_g01_c02.avi,ApplyEyeMakeup,.avi
2,C:\Users\admin\ViTPose_Action_Recognition\data...,ApplyEyeMakeup\v_ApplyEyeMakeup_g01_c03.avi,ApplyEyeMakeup,.avi
3,C:\Users\admin\ViTPose_Action_Recognition\data...,ApplyEyeMakeup\v_ApplyEyeMakeup_g01_c04.avi,ApplyEyeMakeup,.avi
4,C:\Users\admin\ViTPose_Action_Recognition\data...,ApplyEyeMakeup\v_ApplyEyeMakeup_g01_c05.avi,ApplyEyeMakeup,.avi


In [1]:

from pathlib import Path
from collections import Counter

ROOT = Path(r"C:\Users\admin\ViTPose_Action_Recognition")

video_extensions = {".mp4", ".avi", ".mov", ".mkv", ".webm"}
excluded = {
    ".git", "__pycache__", ".ipynb_checkpoints",
    "outputs", "checkpoints", "venv", ".venv"
}

videos = []

for path in ROOT.rglob("*"):
    if not path.is_file() or path.suffix.lower() not in video_extensions:
        continue

    relative = path.relative_to(ROOT)

    if any(part.lower() in excluded for part in relative.parts):
        continue

    videos.append(path)

print("TOTAL ORIGINAL VIDEO FILES:", len(videos))
print("\nVIDEO COUNTS BY PARENT FOLDER:")

counts = Counter(str(p.parent.relative_to(ROOT)) for p in videos)

for folder, count in sorted(counts.items()):
    print(f"{folder}: {count}")

print("\nFIRST 30 VIDEO PATHS:")
for path in sorted(videos)[:30]:
    print(path.relative_to(ROOT))


TOTAL ORIGINAL VIDEO FILES: 13320

VIDEO COUNTS BY PARENT FOLDER:
data\raw\ucf_sports\ApplyEyeMakeup: 145
data\raw\ucf_sports\ApplyLipstick: 114
data\raw\ucf_sports\Archery: 145
data\raw\ucf_sports\BabyCrawling: 132
data\raw\ucf_sports\BalanceBeam: 108
data\raw\ucf_sports\BandMarching: 155
data\raw\ucf_sports\BaseballPitch: 150
data\raw\ucf_sports\Basketball: 134
data\raw\ucf_sports\BasketballDunk: 131
data\raw\ucf_sports\BenchPress: 160
data\raw\ucf_sports\Biking: 134
data\raw\ucf_sports\Billiards: 150
data\raw\ucf_sports\BlowDryHair: 131
data\raw\ucf_sports\BlowingCandles: 109
data\raw\ucf_sports\BodyWeightSquats: 112
data\raw\ucf_sports\Bowling: 155
data\raw\ucf_sports\BoxingPunchingBag: 163
data\raw\ucf_sports\BoxingSpeedBag: 134
data\raw\ucf_sports\BreastStroke: 101
data\raw\ucf_sports\BrushingTeeth: 131
data\raw\ucf_sports\CleanAndJerk: 112
data\raw\ucf_sports\CliffDiving: 138
data\raw\ucf_sports\CricketBowling: 139
data\raw\ucf_sports\CricketShot: 167
data\raw\ucf_sports\Cutting

In [2]:

from pathlib import Path
from collections import defaultdict

ROOT = Path(r"C:\Users\admin\ViTPose_Action_Recognition")

VIDEO_DIR = ROOT / "data" / "raw" / "ucf_sports"
POSE_DIR = ROOT / "data" / "processed" / "keypoints"

CLASSES = [
    "BenchPress",
    "Biking",
    "BoxingPunchingBag",
    "HorseRiding",
    "TennisSwing",
]

video_exts = {".avi", ".mp4", ".mov", ".mkv", ".webm"}

# Index saved pose files by their filename stem
pose_files = list(POSE_DIR.glob("*.npz"))
pose_stems = {p.stem for p in pose_files}

print("Saved NPZ files:", len(pose_files))
print("\nAUDIT BY CLASS")

for cls in CLASSES:
    folder = VIDEO_DIR / cls

    if not folder.exists():
        print(f"\n{cls}: VIDEO FOLDER NOT FOUND: {folder}")
        continue

    videos = sorted(
        p for p in folder.iterdir()
        if p.is_file() and p.suffix.lower() in video_exts
    )

    # Check common naming convention:
    # v_Biking_g01_c01.avi -> Biking_g01_c01
    matched = []
    missing = []

    for video in videos:
        stem = video.stem
        expected = stem[2:] if stem.startswith("v_") else stem

        if expected in pose_stems:
            matched.append(video.name)
        else:
            missing.append(video.name)

    print(f"\n{cls}")
    print("  Original videos:", len(videos))
    print("  Matching NPZ files:", len(matched))
    print("  Not matched by filename:", len(missing))
    print("  Missing examples:", missing[:5])


Saved NPZ files: 13319

AUDIT BY CLASS

BenchPress
  Original videos: 160
  Matching NPZ files: 0
  Not matched by filename: 160
  Missing examples: ['v_BenchPress_g01_c01.avi', 'v_BenchPress_g01_c02.avi', 'v_BenchPress_g01_c03.avi', 'v_BenchPress_g01_c04.avi', 'v_BenchPress_g01_c05.avi']

Biking
  Original videos: 134
  Matching NPZ files: 0
  Not matched by filename: 134
  Missing examples: ['v_Biking_g01_c01.avi', 'v_Biking_g01_c02.avi', 'v_Biking_g01_c03.avi', 'v_Biking_g01_c04.avi', 'v_Biking_g02_c01.avi']

BoxingPunchingBag
  Original videos: 163
  Matching NPZ files: 0
  Not matched by filename: 163
  Missing examples: ['v_BoxingPunchingBag_g01_c01.avi', 'v_BoxingPunchingBag_g01_c02.avi', 'v_BoxingPunchingBag_g01_c03.avi', 'v_BoxingPunchingBag_g01_c04.avi', 'v_BoxingPunchingBag_g01_c05.avi']

HorseRiding
  Original videos: 164
  Matching NPZ files: 0
  Not matched by filename: 164
  Missing examples: ['v_HorseRiding_g01_c01.avi', 'v_HorseRiding_g01_c02.avi', 'v_HorseRiding_g01_c

In [3]:
from pathlib import Path

ROOT = Path(r"C:\Users\admin\ViTPose_Action_Recognition")
POSE_DIR = ROOT / "data" / "processed" / "keypoints"

files = sorted(POSE_DIR.glob("*.npz"))

print("Total NPZ files:", len(files))
print("\nFirst 15 filenames:")
for f in files[:15]:
    print(f.name)

Total NPZ files: 13319

First 15 filenames:
ApplyEyeMakeup__v_ApplyEyeMakeup_g01_c04__e2cb58767c.npz
ApplyEyeMakeup__v_ApplyEyeMakeup_g01_c05__85feeac362.npz
ApplyEyeMakeup__v_ApplyEyeMakeup_g02_c01__560a140366.npz
ApplyEyeMakeup__v_ApplyEyeMakeup_g02_c02__fe57b9530b.npz
ApplyEyeMakeup__v_ApplyEyeMakeup_g02_c03__1adcc92044.npz
ApplyEyeMakeup__v_ApplyEyeMakeup_g02_c04__6f6b2def3a.npz
ApplyEyeMakeup__v_ApplyEyeMakeup_g03_c01__33dbf8fc73.npz
ApplyEyeMakeup__v_ApplyEyeMakeup_g03_c02__ac57e00f6e.npz
ApplyEyeMakeup__v_ApplyEyeMakeup_g03_c03__508b0a3b8a.npz
ApplyEyeMakeup__v_ApplyEyeMakeup_g03_c04__d24fb69143.npz
ApplyEyeMakeup__v_ApplyEyeMakeup_g03_c05__dfe0acf3fc.npz
ApplyEyeMakeup__v_ApplyEyeMakeup_g03_c06__67fb20d7fd.npz
ApplyEyeMakeup__v_ApplyEyeMakeup_g04_c01__b644f78057.npz
ApplyEyeMakeup__v_ApplyEyeMakeup_g04_c02__a49023f2b1.npz
ApplyEyeMakeup__v_ApplyEyeMakeup_g04_c03__23fc28bcf5.npz
